In [ ]:
!mkdir -p /content/finetuning
!mkdir -p /content/data/processed

In [ ]:
import torch
!nvidia-smi
print("\nPyTorch :", torch.__version__)
print("CUDA    :", torch.version.cuda)

Sat Jun 27 09:20:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
%%capture
!pip install "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade unsloth_zoo

In [ ]:
%%capture
!pip install \
    "trl>=0.24.0" \
    "transformers>=4.57.6" \
    "peft>=0.19.1" \
    datasets \
    accelerate \
    pyyaml \
    wandb

In [ ]:
import unsloth, trl, transformers, peft, bitsandbytes as bnb
print("unsloth     :", unsloth.__version__)
print("trl         :", trl.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("bitsandbytes:", bnb.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth     : 2026.6.9
trl         : 0.24.0
transformers: 5.5.0
peft        : 0.19.1
bitsandbytes: 0.49.2


In [ ]:
import os

for d in ["/content/outputs/qwen-tax-lora"]:
    os.makedirs(d, exist_ok=True)
    print(f"✅  {d}")

✅  /content/outputs/qwen-tax-lora


In [ ]:
%%writefile /content/finetuning/qlora_config.yaml
# QLoRA configuration
model:
  name: "Qwen/Qwen3-8B"
  trust_remote_code: true

dataset:
  train_file: "data/processed/finetune_train.jsonl"
  eval_file: "data/processed/finetune_eval.jsonl"

quantization:
  load_in_4bit: true
  bnb_4bit_quant_type: "nf4"
  bnb_4bit_compute_dtype: "bfloat16"
  bnb_4bit_use_double_quant: true

lora:
  r: 64
  lora_alpha: 128
  lora_dropout: 0.05
  target_modules:                  # explicit list — avoids PEFT string-iteration bug
    - q_proj
    - k_proj
    - v_proj
    - o_proj
    - gate_proj
    - up_proj
    - down_proj
  use_dora: true
  bias: "none"

training:
  output_dir: "outputs/qwen-tax-lora"
  num_train_epochs: 6
  per_device_train_batch_size: 16
  per_device_eval_batch_size: 16
  gradient_accumulation_steps: 16
  learning_rate: 0.0002
  warmup_ratio: 0.03
  lr_scheduler_type: "cosine"
  optim: "paged_adamw_8bit"
  weight_decay: 0.01
  max_grad_norm: 1.0
  bf16: true
  logging_steps: 10
  save_strategy: "epoch"
  evaluation_strategy: "epoch"
  save_total_limit: 2
  load_best_model_at_end: true
  report_to: "wandb"

Overwriting /content/finetuning/qlora_config.yaml


In [ ]:
%%writefile /content/finetuning/train_qlora.py
"""
finetuning/train_qlora.py
QLoRA fine-tuning for Qwen3-8B (base model) on legal Q&A data.
"""

import os
import sys
import yaml

from datasets import load_dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, SFTConfig


# ─────────────────────────────────────────────
# 1. Config
# ─────────────────────────────────────────────

def load_config(path: str = "finetuning/qlora_config.yaml") -> dict:
    if not os.path.exists(path):
        sys.exit(f"[ERROR] Config not found: {path}")
    with open(path, "r", encoding="utf-8") as fh:
        return yaml.safe_load(fh)


# ─────────────────────────────────────────────
# 2. Dataset
# ─────────────────────────────────────────────

def load_datasets(config: dict):
    for key in ("train_file", "eval_file"):
        if not os.path.exists(config["dataset"][key]):
            sys.exit(f"[ERROR] Dataset not found: {config['dataset'][key]}")

    train_ds = load_dataset("json", data_files=config["dataset"]["train_file"], split="train")
    eval_ds  = load_dataset("json", data_files=config["dataset"]["eval_file"],  split="train")

    print(f"  Train samples : {len(train_ds)}")
    print(f"  Eval  samples : {len(eval_ds)}")
    print(f"  Sample keys   : {list(train_ds[0].keys())}")
    return train_ds, eval_ds


def apply_chat_template(dataset, tokenizer):
    def _format(example):
        return {
            "text": tokenizer.apply_chat_template(
                example["messages"],
                tokenize=False,
                add_generation_prompt=False,
            )
        }
    return dataset.map(_format, remove_columns=dataset.column_names,
                       desc="Applying chat template")


# ─────────────────────────────────────────────
# 3. Model + tokenizer
# ─────────────────────────────────────────────

def load_model(config: dict):
    q = config["quantization"]

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name        = config["model"]["name"],
        max_seq_length    = 4096,
        dtype             = None,
        load_in_4bit      = q["load_in_4bit"],
        trust_remote_code = config["model"]["trust_remote_code"],
    )

    # Set chat template explicitly for base model
    for template in ("qwen-3", "qwen-2.5"):
        try:
            tokenizer = get_chat_template(tokenizer, chat_template=template)
            print(f"  Chat template  : {template} (via Unsloth)")
            break
        except Exception:
            continue
    else:
        print("  Chat template  : using tokenizer built-in")

    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


# ─────────────────────────────────────────────
# 4. LoRA / DoRA
# ─────────────────────────────────────────────

def attach_lora(model, config: dict):
    lora = config["lora"]

    target = lora["target_modules"]

    # Guard: PEFT 0.19+ iterates a bare string as characters → explodes.
    # If YAML still has "all-linear" string, convert to explicit list.
    if isinstance(target, str):
        print(f"  WARNING: target_modules is a string '{target}' — converting to explicit list")
        target = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

    print(f"  Target modules : {target}")

    model = FastLanguageModel.get_peft_model(
        model,
        r              = lora["r"],
        target_modules = target,
        lora_alpha     = lora["lora_alpha"],
        lora_dropout   = lora["lora_dropout"],
        bias           = lora["bias"],
        use_dora       = lora["use_dora"],
        use_gradient_checkpointing = "unsloth",
        random_state   = 42,
    )
    model.print_trainable_parameters()
    return model


# ─────────────────────────────────────────────
# 5. Trainer
# ─────────────────────────────────────────────

def build_trainer(model, tokenizer, train_ds, eval_ds, config: dict):
    t = config["training"]

    sft_config = SFTConfig(
        output_dir                  = t["output_dir"],
        num_train_epochs            = t["num_train_epochs"],
        per_device_train_batch_size = t["per_device_train_batch_size"],
        per_device_eval_batch_size  = t["per_device_eval_batch_size"],
        gradient_accumulation_steps = t["gradient_accumulation_steps"],
        learning_rate               = float(t["learning_rate"]),
        optim                       = t["optim"],
        weight_decay                = t["weight_decay"],
        max_grad_norm               = t["max_grad_norm"],
        warmup_ratio                = t["warmup_ratio"],
        lr_scheduler_type           = t["lr_scheduler_type"],
        bf16                        = t["bf16"],
        fp16                        = False,
        eval_strategy               = t["evaluation_strategy"],
        save_strategy               = t["save_strategy"],
        save_total_limit            = t["save_total_limit"],
        load_best_model_at_end      = t["load_best_model_at_end"],
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        logging_steps               = t["logging_steps"],
        report_to                   = t["report_to"],
        max_seq_length              = 4096,
        dataset_text_field          = "text",
        packing                     = False,
    )

    return SFTTrainer(
        model            = model,
        processing_class = tokenizer,
        train_dataset    = train_ds,
        eval_dataset     = eval_ds,
        args             = sft_config,
    )


# ─────────────────────────────────────────────
# 6. Main
# ─────────────────────────────────────────────

def main():
    config_path = os.environ.get("QLORA_CONFIG", "finetuning/qlora_config.yaml")

    print(f"\n{'='*62}")
    print(f"  QLoRA Fine-Tuning  |  Qwen3-8B (base)  |  A100")
    print(f"  Config: {config_path}")
    print(f"{'='*62}\n")

    config = load_config(config_path)
    os.makedirs(config["training"]["output_dir"], exist_ok=True)

    print("[1/5]  Loading datasets ...")
    train_ds, eval_ds = load_datasets(config)

    print("\n[2/5]  Loading Qwen3-8B in 4-bit (NF4) ...")
    model, tokenizer = load_model(config)

    print("\n[3/5]  Injecting LoRA / DoRA adapters ...")
    model = attach_lora(model, config)

    print("\n[4/5]  Applying chat template to datasets ...")
    train_ds = apply_chat_template(train_ds, tokenizer)
    eval_ds  = apply_chat_template(eval_ds,  tokenizer)
    print(f"  Sample (first 200 chars): {train_ds[0]['text'][:200]!r}")

    eff_batch = (config["training"]["per_device_train_batch_size"]
                 * config["training"]["gradient_accumulation_steps"])
    print(f"\n[5/5]  Starting training ...  (effective batch = {eff_batch})\n")

    trainer = build_trainer(model, tokenizer, train_ds, eval_ds, config)
    trainer.train()

    adapter_dir = os.path.join(config["training"]["output_dir"], "final_adapter")
    os.makedirs(adapter_dir, exist_ok=True)
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)

    print(f"\n{'='*62}")
    print(f"  Training complete.")
    print(f"  Adapter saved -> {adapter_dir}")
    print(f"{'='*62}\n")


if __name__ == "__main__":
    main()

Overwriting /content/finetuning/train_qlora.py


In [ ]:
print("Upload your 2 JSONL files into /content/data/processed/ using the 📁 left panel, then run Cell 9.")

Upload your 2 JSONL files into /content/data/processed/ using the 📁 left panel, then run Cell 9.


In [ ]:
import os

required = {
    "train_qlora.py"       : "/content/finetuning/train_qlora.py",
    "qlora_config.yaml"    : "/content/finetuning/qlora_config.yaml",
    "finetune_train.jsonl" : "/content/data/processed/finetune_train.jsonl",
    "finetune_eval.jsonl"  : "/content/data/processed/finetune_eval.jsonl",
}

all_ok = True
for name, path in required.items():
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"✅  {name:30s}  {size_kb:>8.1f} KB")
    else:
        print(f"❌  {name:30s}  MISSING")
        all_ok = False

print()
print("🟢 Ready to train!" if all_ok else "🔴 Fix missing files before continuing.")

✅  train_qlora.py                       8.3 KB
✅  qlora_config.yaml                    1.1 KB
✅  finetune_train.jsonl              5323.0 KB
✅  finetune_eval.jsonl               1319.5 KB

🟢 Ready to train!


In [ ]:
from huggingface_hub import login

import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)
print("✅ Logged into HuggingFace")

✅ Logged into HuggingFace


In [ ]:
import wandb
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kk9328781 (kk9328781-indian-institute-of-information-technology-sricity) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
%cd /content
!python finetuning/train_qlora.py

/content
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

  QLoRA Fine-Tuning  |  Qwen3-8B (base)  |  A100
  Config: finetuning/qlora_config.yaml

[1/5]  Loading datasets ...
  Train samples : 2945
  Eval  samples : 737
  Sample keys   : ['messages', 'category']

[2/5]  Loading Qwen3-8B in 4-bit (NF4) ...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 399/399 [00:02<00:0

In [ ]:
import os

adapter_dir = "/content/outputs/qwen-tax-lora/final_adapter"

if os.path.exists(adapter_dir):
    total_mb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, fnames in os.walk(adapter_dir)
        for f in fnames
    ) / 1024 / 1024
    print(f"✅ Adapter saved  ({total_mb:.1f} MB total)\n")
    for fname in sorted(os.listdir(adapter_dir)):
        size = os.path.getsize(os.path.join(adapter_dir, fname)) / 1024 / 1024
        print(f"  {fname:50s}  {size:.1f} MB")
else:
    print("❌ Adapter not found — check training logs above.")

✅ Adapter saved  (682.4 MB total)

  README.md                                           0.0 MB
  adapter_config.json                                 0.0 MB
  adapter_model.safetensors                           671.4 MB
  chat_template.jinja                                 0.0 MB
  tokenizer.json                                      10.9 MB
  tokenizer_config.json                               0.0 MB


In [ ]:
import gc
import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

gc.collect()
torch.cuda.empty_cache()

adapter_path = "/content/outputs/qwen-tax-lora/final_adapter"
merged_path  = "/content/outputs/qwen-tax-merged"

print("1/4  Loading Qwen3-8B base in FP16 ...")
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-8B",
    torch_dtype       = torch.float16,
    device_map        = "auto",
    trust_remote_code = True,
    token             = HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-8B",
    trust_remote_code = True,
    token             = HF_TOKEN,
)

print("2/4  Attaching DoRA adapter ...")
peft_model = PeftModel.from_pretrained(base_model, adapter_path)

print("3/4  Merging DoRA weights ...")
merged = peft_model.merge_and_unload()

print("4/4  Saving merged FP16 model ...")
merged.save_pretrained(merged_path, safe_serialization=True, max_shard_size="4GB")
tokenizer.save_pretrained(merged_path)

del merged, peft_model, base_model
gc.collect()
torch.cuda.empty_cache()

print(f"\n✅ Merged model saved → {merged_path}")

1/4  Loading Qwen3-8B base in FP16 ...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

2/4  Attaching DoRA adapter ...
3/4  Merging DoRA weights ...
4/4  Saving merged FP16 model ...


Writing model shards:   0%|          | 0/5 [00:00<?, ?it/s]


✅ Merged model saved → /content/outputs/qwen-tax-merged


In [ ]:
import os
import shutil
import gc
import torch

# ── 1. Delete HuggingFace model cache (~16 GB) ────────────────
hf_cache = os.path.expanduser("~/.cache/huggingface/hub")
if os.path.exists(hf_cache):
    shutil.rmtree(hf_cache)
    print("✅ HuggingFace cache deleted")

# ── 2. Delete training checkpoints (keep only final_adapter) ──
checkpoint_dir = "/content/outputs/qwen-tax-lora"
for item in os.listdir(checkpoint_dir):
    if item.startswith("checkpoint-"):
        shutil.rmtree(os.path.join(checkpoint_dir, item))
        print(f"✅ Deleted {item}")

# ── 3. Clear GPU + RAM ─────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()

# ── 4. Check free disk now ─────────────────────────────────────
stat = shutil.disk_usage("/content")
free_gb  = stat.free  / 1024**3
total_gb = stat.total / 1024**3
used_gb  = stat.used  / 1024**3
print(f"\nDisk after cleanup:")
print(f"  Used  : {used_gb:.1f} GB")
print(f"  Free  : {free_gb:.1f} GB")
print(f"  Total : {total_gb:.1f} GB")
print()
if free_gb > 6:
    print("🟢 Enough space — safe to run GGUF cell now.")
else:
    print("🔴 Still low — also delete merged model and re-merge after GGUF.")

✅ HuggingFace cache deleted

Disk after cleanup:
  Used  : 74.0 GB
  Free  : 38.7 GB
  Total : 112.6 GB

🟢 Enough space — safe to run GGUF cell now.


In [ ]:
# Mount Google Drive — much more reliable than files.download() for large files
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("/content/drive/MyDrive/qwen_tax_model", exist_ok=True)
print("✅ Google Drive mounted → /content/drive/MyDrive/qwen_tax_model/")

Mounted at /content/drive
✅ Google Drive mounted → /content/drive/MyDrive/qwen_tax_model/


In [ ]:
# ── GGUF via Unsloth skipped ──────────────────────────────────────────────
# Unsloth 2026.6.x has a bug with DoRA (lora_magnitude_vector keys) inside
# save_pretrained_gguf. The correct GGUF path is via llama.cpp (Cell 25).
# Skip this cell and jump directly to Cell 25.
print("⏭  Skipping Unsloth GGUF cell — use the llama.cpp cell below instead.")


⏭  Skipping Unsloth GGUF cell — use the llama.cpp cell below instead.


In [ ]:
import os, shutil, gc, torch

def delete(path, label):
    if os.path.exists(path):
        size = sum(os.path.getsize(os.path.join(dp,f))
                   for dp,_,fnames in os.walk(path)
                   for f in fnames)/1024**3 if os.path.isdir(path) \
               else os.path.getsize(path)/1024**3
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
        print(f"  ✅ {label:40s}  freed {size:.1f} GB")
    else:
        print(f"  ⬜ {label:40s}  not found")

delete("/content/outputs/qwen-tax-merged",              "Merged FP16 model")
delete("/content/outputs/qwen-tax-gguf",                "Broken empty GGUF")
delete(os.path.expanduser("~/.cache/huggingface/hub"), "HuggingFace cache")
delete(os.path.expanduser("~/.cache/pip"),              "pip cache")
delete(os.path.expanduser("~/.triton"),                 "Triton cache")
delete(os.path.expanduser("~/.cache/torch"),            "Torch cache")

for item in os.listdir("/content/outputs/qwen-tax-lora"):
    if item.startswith("checkpoint-"):
        delete(f"/content/outputs/qwen-tax-lora/{item}", f"Checkpoint {item}")

gc.collect()
torch.cuda.empty_cache()

stat = shutil.disk_usage("/content")
print(f"\n  Free: {stat.free/1024**3:.1f} GB  |  Used: {stat.used/1024**3:.1f} GB")

  ✅ Merged FP16 model                         freed 15.3 GB
  ✅ Broken empty GGUF                         freed 0.0 GB
  ⬜ HuggingFace cache                         not found
  ✅ pip cache                                 freed 0.2 GB
  ⬜ Triton cache                              not found
  ⬜ Torch cache                               not found

  Free: 54.1 GB  |  Used: 58.5 GB


In [ ]:
import gc, torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

gc.collect()
torch.cuda.empty_cache()

print("1/4  Loading Qwen3-8B base in FP16 ...")
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-8B",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-8B", trust_remote_code=True, token=HF_TOKEN
)

print("2/4  Attaching DoRA adapter ...")
peft_model = PeftModel.from_pretrained(
    base_model, "/content/outputs/qwen-tax-lora/final_adapter"
)

print("3/4  Merging DoRA weights ...")
merged = peft_model.merge_and_unload()

print("4/4  Saving merged FP16 model ...")
os.makedirs("/content/outputs/qwen-tax-merged", exist_ok=True)
merged.save_pretrained(
    "/content/outputs/qwen-tax-merged",
    safe_serialization=True,
    max_shard_size="4GB"
)
tokenizer.save_pretrained("/content/outputs/qwen-tax-merged")



1/4  Loading Qwen3-8B base in FP16 ...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

2/4  Attaching DoRA adapter ...
3/4  Merging DoRA weights ...
4/4  Saving merged FP16 model ...


Writing model shards:   0%|          | 0/5 [00:00<?, ?it/s]

('/content/outputs/qwen-tax-merged/tokenizer_config.json',
 '/content/outputs/qwen-tax-merged/chat_template.jinja',
 '/content/outputs/qwen-tax-merged/tokenizer.json')

In [ ]:
import os, shutil

print("Merged model exists:", os.path.exists("/content/outputs/qwen-tax-merged"))
print("Final adapter exists:", os.path.exists("/content/outputs/qwen-tax-lora/final_adapter"))

stat = shutil.disk_usage("/content")
print(f"Free disk: {stat.free/1024**3:.1f} GB")
print(f"Used disk: {stat.used/1024**3:.1f} GB")

Merged model exists: True
Final adapter exists: True
Free disk: 24.7 GB
Used disk: 88.0 GB


In [ ]:
import os
import shutil
import json as _json

# ── 1. Install llama.cpp ────────────────────────────────────────
!pip -q install gguf sentencepiece protobuf

if os.path.exists("/content/llama.cpp"):
    shutil.rmtree("/content/llama.cpp")

!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install -r /content/llama.cpp/requirements.txt

# ── 2. Build llama.cpp ──────────────────────────────────────────
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON
!cmake --build /content/llama.cpp/build --config Release -j$(nproc)

os.makedirs("/content/outputs/qwen-tax-gguf", exist_ok=True)

# ── 2b. Patch config.json ───────────────────────────────────────
_cfg_path = "/content/outputs/qwen-tax-merged/config.json"
with open(_cfg_path) as _f:
    _cfg = _json.load(_f)
if _cfg.get("model_type") != "qwen3":
    _cfg["model_type"] = "qwen3"
    with open(_cfg_path, "w") as _f:
        _json.dump(_cfg, _f, indent=2)
    print("✅ Patched config.json model_type -> qwen3")
else:
    print("✅ config.json model_type already qwen3")

# ── 2c. Patch tokenizer_config.json ────────────────────────────
_tok_cfg_path = "/content/outputs/qwen-tax-merged/tokenizer_config.json"
with open(_tok_cfg_path) as _f:
    _tok_cfg = _json.load(_f)
if isinstance(_tok_cfg.get("extra_special_tokens"), list):
    _tok_cfg["extra_special_tokens"] = {}
    with open(_tok_cfg_path, "w") as _f:
        _json.dump(_tok_cfg, _f, indent=2)
    print("✅ Patched tokenizer_config.json extra_special_tokens")
else:
    print("✅ tokenizer_config.json already fine")

# ── 3. Convert to F16 GGUF (convert_hf_to_gguf only supports f16/f32) ──
f16_path = "/content/outputs/qwen-tax-gguf/model-f16.gguf"
print("\nStep 1/2: Converting merged model → F16 GGUF ...")

!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/outputs/qwen-tax-merged \
    --outfile {f16_path} \
    --outtype f16

assert os.path.exists(f16_path) and os.path.getsize(f16_path) > 1e9, \
    "F16 GGUF conversion failed or file too small."

size_f16 = os.path.getsize(f16_path) / 1024**3
print(f"✅ F16 GGUF: {size_f16:.2f} GB")

# ── 4. Delete merged model now to free space ────────────────────
if os.path.exists("/content/outputs/qwen-tax-merged"):
    shutil.rmtree("/content/outputs/qwen-tax-merged")
    print("✅ Deleted merged model")

stat = shutil.disk_usage("/content")
print(f"Free disk after deleting merged: {stat.free/1024**3:.1f} GB")

# ── 5. Quantize F16 → Q4_K_M (llama-quantize accepts F16 input) ──
final_path = "/content/outputs/qwen-tax-gguf/qwen3-8b-tax-q4km.gguf"
print("\nStep 2/2: Quantizing F16 → Q4_K_M ...")

!/content/llama.cpp/build/bin/llama-quantize \
    {f16_path} \
    {final_path} \
    Q4_K_M

assert os.path.exists(final_path) and os.path.getsize(final_path) > 1e9, \
    "Q4_K_M quantization failed or file too small."

# ── 6. Delete F16 intermediate ──────────────────────────────────
os.remove(f16_path)
print("✅ Deleted F16 intermediate")

stat = shutil.disk_usage("/content")
size = os.path.getsize(final_path) / 1024**3
print(f"\n✅ GGUF ready: {final_path}")
print(f"Size: {size:.2f} GB")
print(f"Free disk: {stat.free/1024**3:.1f} GB")

Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 3330, done.
remote: Counting objects: 100% (3330/3330), done.
remote: Compressing objects: 100% (2683/2683), done.
remote: Total 3330 (delta 613), reused 2747 (delta 565), pack-reused 0 (from 0)
Receiving objects: 100% (3330/3330), 33.46 MiB | 20.08 MiB/s, done.
Resolving deltas: 100% (613/613), done.
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Foun

In [ ]:
import os

gguf_dir = "/content/outputs/qwen-tax-gguf"
print("Files in GGUF folder:")
if os.path.exists(gguf_dir):
    files = os.listdir(gguf_dir)
    if files:
        for f in files:
            size = os.path.getsize(os.path.join(gguf_dir, f)) / 1024**2
            print(f"  {f}  →  {size:.1f} MB")
    else:
        print("  ❌ Folder is empty — GGUF failed mid-write (out of disk)")
else:
    print("  ❌ Folder doesn't exist")

Files in GGUF folder:
  qwen3-8b-tax-q4km.gguf  →  4794.9 MB


In [ ]:
import os, shutil
from google.colab import files

print("Zipping GGUF model ...")
shutil.make_archive("/content/qwen_tax_gguf", "zip", "/content/outputs/qwen-tax-gguf")

size_gb = os.path.getsize("/content/qwen_tax_gguf.zip") / 1024 / 1024 / 1024
print(f"✅ Archive ready: qwen_tax_gguf.zip  ({size_gb:.1f} GB)")
print("Starting download ...")
files.download("/content/qwen_tax_gguf.zip")

Zipping GGUF model ...
✅ Archive ready: qwen_tax_gguf.zip  (4.6 GB)
Starting download ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>